## Install Dependencies

In [ ]:
!pip install -q transformers==4.51.3 autoawq "langchain<1.0.0" "langchain-community<1.0.0" langchain-groq \
langchain-huggingface langchain-openai langgraph faiss-cpu flashrank FlagEmbedding \
pypdf sentence-transformers tavily-python markdown deepeval openpyxl langchain-core rank_bm25

## Setup & Authentication

In [1]:
import os
from kaggle_secrets import UserSecretsClient

# disable the buggy Xet protocol and fall back to the standard, stable HTTP download method.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Securely load the API keys using Kaggle Secrets
user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
# if rate limit hit use this
# os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_EVAL_API_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["TAVILY_API_KEY"] = user_secrets.get_secret("TAVILY_API_KEY")

print("✅ API Keys loaded successfully.")

✅ API Keys loaded successfully.


## Ingestion & Vector Database

In [2]:
import pickle
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# Configuration Paths
DATASET_PATH = "/kaggle/input/datasets/ashishmm10/irdaidataset"
WORKING_INDEX_PATH = "/kaggle/working/faiss_index"
CHUNK_CACHE_PATH = "/kaggle/working/all_chunks.pkl"

# 1. Initialize embeddings
print("Initializing BGE-Large Embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda:1'}
)

FORCE_REBUILD = True  # ⚠️ Set True to rebuild index after chunk changes

# ==========================================
# 2. Always load raw documents
#    ParentDocumentRetriever uses InMemoryStore
#    which is not persisted — must rebuild every session.
# ==========================================
print("🔄 Loading PDFs (always required for ParentDocumentRetriever)...")
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"PDF directory not found at {DATASET_PATH}.")

loader = PyPDFDirectoryLoader(DATASET_PATH)
documents = loader.load()
print(f"   Loaded {len(documents)} raw PDF pages.")

# ==========================================
# 3. Define splitters
# ==========================================
# Child splitter — small chunks indexed in FAISS for precise retrieval
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)

# Parent splitter — larger chunks returned to the LLM for rich context
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)

# ==========================================
# 4. BM25 parent chunks (cached)
# ==========================================
if not FORCE_REBUILD and os.path.exists(CHUNK_CACHE_PATH):
    print("✅ Found cached parent chunks. Loading for BM25...")
    with open(CHUNK_CACHE_PATH, "rb") as f:
        all_chunks = pickle.load(f)
    print(f"   Loaded {len(all_chunks)} parent chunks.")
else:
    print("🔄 Building parent chunks for BM25...")
    all_chunks = parent_splitter.split_documents(documents)

    # Prepend document title to each parent chunk
    # Helps BM25 and embeddings disambiguate across circulars/acts
    for chunk in all_chunks:
        source = chunk.metadata.get("source", "")
        doc_title = os.path.basename(source).replace(".pdf", "").replace("_", " ")
        chunk.page_content = f"[Document: {doc_title}]\n{chunk.page_content}"

    print(f"✅ {len(all_chunks)} parent chunks built and title-prepended.")

    with open(CHUNK_CACHE_PATH, "wb") as f:
        pickle.dump(all_chunks, f)
    print(f"   Cached to {CHUNK_CACHE_PATH}.")

# ==========================================
# 5. FAISS index (cached)
# ==========================================
if not FORCE_REBUILD and os.path.exists(WORKING_INDEX_PATH):
    print(f"✅ Found session FAISS index. Loading...")
    vectorstore = FAISS.load_local(
        WORKING_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
    )
else:
    vectorstore = None

if vectorstore is None:
    print("🔄 Building FAISS index from scratch (this may take a moment)...")
    # FAISS is seeded with parent chunks here so it is consistent with BM25.
    # ParentDocumentRetriever will add child chunks into it next.
    vectorstore = FAISS.from_documents(all_chunks, embeddings)
    vectorstore.save_local(WORKING_INDEX_PATH)
    print(f"✅ FAISS index built and saved to {WORKING_INDEX_PATH}.")

# ==========================================
# 6. ParentDocumentRetriever
#    — child chunks go into FAISS for retrieval
#    — matched child chunks are expanded to parent chunks for LLM context
#    — InMemoryStore is always rebuilt (not persisted)
# ==========================================
print("🔄 Building ParentDocumentRetriever (always rebuilt — InMemoryStore is ephemeral)...")
docstore = InMemoryStore()
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_retriever.add_documents(documents)
print(f"✅ ParentDocumentRetriever ready — child chunks in FAISS, parents in docstore.")

Initializing BGE-Large Embeddings...


2026-04-12 05:15:55.599811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775970956.092349     148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775970956.214775     148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775970957.284777     148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775970957.284811     148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775970957.284813     148 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🔄 Loading PDFs (always required for ParentDocumentRetriever)...
   Loaded 699 raw PDF pages.
🔄 Building parent chunks for BM25...
✅ 1521 parent chunks built and title-prepended.
   Cached to /kaggle/working/all_chunks.pkl.
🔄 Building FAISS index from scratch (this may take a moment)...
✅ FAISS index built and saved to /kaggle/working/faiss_index.
🔄 Building ParentDocumentRetriever (always rebuilt — InMemoryStore is ephemeral)...
✅ ParentDocumentRetriever ready — child chunks in FAISS, parents in docstore.


## LangGraph Architecture

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from sentence_transformers import CrossEncoder as RawCrossEncoder
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage, HumanMessage

# ═══════════════════════════════════════════════════════════════════
# 🔴 IMPROVEMENT: BM25 Hybrid Search (keyword + semantic)
# ═══════════════════════════════════════════════════════════════════
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# ==========================================
# 1. Initialize Tools & LLMs
# ==========================================
primary_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

router_llm = ChatGroq(
    model="llama-3.3-70b-versatile", # Now using stronger model for routing decision
    temperature=0,
    max_retries=2
)

# ── Query-aware BM25/FAISS weight selector ──────────────────────
import re as _re

_CLAUSE_PATTERN = _re.compile(
    r"section\s+\d|clause\s+\d|regulation\s+\d|rule\s+\d|schedule\s+\d|"
    r"chapter\s+\d|sub-?\s*section|paragraph\s+\d|article\s+\d",
    _re.IGNORECASE,
)

def get_retriever_weights(query: str) -> tuple[float, float]:
    """Return (bm25_weight, faiss_weight) based on query specificity."""
    if _CLAUSE_PATTERN.search(query):
        print("   ⚖️  Clause-specific query → BM25 0.6 / FAISS 0.4")
        return 0.6, 0.4
    print("   ⚖️  Semantic query → BM25 0.35 / FAISS 0.65")
    return 0.35, 0.65

# BM25 on parent chunks (richer keyword surface for regulatory clauses)
bm25_retriever = BM25Retriever.from_documents(all_chunks, k=22)

# FAISS with MMR — child chunks indexed, parent chunks returned via ParentDocumentRetriever
# Use parent_retriever directly as the semantic leg of the ensemble
parent_retriever.search_kwargs = {"k": 22, "fetch_k": 44, "lambda_mult": 0.75}

# Static ensemble with balanced default weights (overridden per-query in retrieve_and_rerank)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.35, 0.65]
)

# ── Upgraded reranker: BGE-reranker-large ───────────────────────
_raw_reranker = RawCrossEncoder(
    "BAAI/bge-reranker-large",
    device="cuda:1",
    max_length=512,
)
hf_reranker = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-large",
    model_kwargs={"device": "cuda:1"},
)
compressor = CrossEncoderReranker(model=hf_reranker, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)
print("✅ Hybrid retriever ready: BM25(k=22) + ParentDoc-FAISS(k=22) → BGE-reranker-large(top_n=5)")

# ── Tiered Domain Strategy ───────────────────────────────────────
REGULATORY_DOMAINS = [
    "irdai.gov.in", "pib.gov.in", "finmin.nic.in", "egazette.gov.in",
]

INSURER_DOMAIN_MAP = {
    "care health": "careinsurance.com", "star health": "starhealth.in",
    "niva bupa": "nivabupa.com", "hdfc ergo": "hdfcergo.com",
    "bajaj allianz": "bajajallianz.com", "icici lombard": "icicilombard.com",
    "manipal cigna": "manipalcigna.com", "new india": "newindia.co.in",
    "united india": "uiic.co.in", "national insurance": "nationalinsurance.nic.co.in",
    "aditya birla": "adityabirlacapital.com", "sbi general": "sbigeneral.in",
    "reliance health": "reliancegeneral.co.in", "tata aig": "tataaig.com",
    "max bupa": "nivabupa.com", "sbi life": "sbilife.co.in",
}

INSURER_DISCLOSURE_DOMAINS = [
    "hdfclife.com", "iciciprulife.com", "maxlifeinsurance.com",
    "starhealth.in", "careinsurance.com", "acko.com", "sbilife.co.in",
    "licindia.in", "icicilombard.com", "hdfcergo.com", "nivabupa.com",
    "manipalcigna.com", "newindia.co.in", "orientalinsurance.org.in",
    "uiic.co.in", "nationalinsurance.nic.co.in",
    "lexology.com", "veritaslegal.in",  # Legal analysis sites
]

BLOCKED_DOMAINS = [
    "policybazaar.com", "coverfox.com", "insurancedekho.com",
    "bankbazaar.com", "reddit.com", "quora.com", "medium.com",
]

def build_tavily_search(query: str) -> TavilySearchResults:
    q = query.lower()
    regulatory_terms = [
        "regulation", "circular", "compliance", "compliant", "schedule",
        "master circular", "chapter", "section", "rule", "act",
        "guideline", "direction", "notification", "gazette",
        "irdai", "clause", "obligation"
    ]
    insurer_disclosure_terms = [
        "annual report", "annual-report", "board committee", "board committees",
        "corporate governance", "governance report", "public disclosure",
        "publicly disclosed", "disclosure", "investor relations",
        "committee structure", "how are leading insurers", "how are insurers",
        "leading insurers", "major insurers", "top insurers", "private insurers"
    ]
    current_status_terms = [
        "latest", "current", "as of", "recent", "recently",
        "who is the current", "has been launched", "latest status",
        "rollout status", "appointed", "chairperson", "ceo"
    ]
    named_insurer_domains = [
        domain for name, domain in INSURER_DOMAIN_MAP.items() if name in q
    ]
    is_regulatory = any(term in q for term in regulatory_terms)
    wants_insurer_disclosure = any(term in q for term in insurer_disclosure_terms)
    wants_current_status = any(term in q for term in current_status_terms)

    if named_insurer_domains:
        domains = list(set(named_insurer_domains + REGULATORY_DOMAINS))
        print(f"   🏢 Domain tier: NAMED INSURER + REGULATORY → {domains}")
        return TavilySearchResults(max_results=6, include_domains=domains)

    if is_regulatory and wants_insurer_disclosure:
        domains = list(set(REGULATORY_DOMAINS + INSURER_DISCLOSURE_DOMAINS))
        print(f"   📄 Domain tier: INSURER DISCLOSURE + REGULATORY → {domains}")
        return TavilySearchResults(max_results=8, include_domains=domains)

    if is_regulatory or wants_current_status:
        print(f"   🔒 Domain tier: REGULATORY ONLY → {REGULATORY_DOMAINS}")
        return TavilySearchResults(max_results=5, include_domains=REGULATORY_DOMAINS)

    print("   🌐 Domain tier: OPEN WEB (blocking aggregators)")
    return TavilySearchResults(max_results=6, exclude_domains=BLOCKED_DOMAINS)


# ==========================================
# 2. State Definition
# ==========================================
class AgentState(TypedDict):
    query: str
    chat_history: str
    route: str
    router_raw_output: str
    router_confidence: float
    fallback_used: bool
    fallback_reason: str
    retrieved_docs: list
    web_results: list
    web_context: str
    final_answer: str


# ==========================================
# 3. Nodes
# ==========================================
import re

def detect_route_confidence(raw_text: str) -> tuple[str, float]:
    text = raw_text.strip()
    route_match = re.search(r"ROUTE\s*:\s*(RAG|WEB|BOTH)", text, re.IGNORECASE)
    conf_match = re.search(r"CONFIDENCE\s*:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    route = route_match.group(1).upper() if route_match else "BOTH"
    confidence = float(conf_match.group(1)) if conf_match else 0.0
    if route not in {"RAG", "WEB", "BOTH"}:
        route = "BOTH"
    confidence = max(0.0, min(confidence, 1.0))
    return route, confidence


def intent_router(state: AgentState):
    system_prompt = f"""
You are a routing assistant for an IRDAI regulatory QA system.

Choose exactly one route:
- RAG  = answer should come from the local IRDAI corpus only
- WEB  = answer requires fresh/current/public web information
- BOTH = answer genuinely needs both the corpus and live web evidence

Routing rules:
1. Use RAG when the question asks about:
   - sections, clauses, schedules, regulations, circulars, guidelines
   - definitions, duties, timelines, penalties, committee requirements
   - content already contained in IRDAI PDFs or legal corpus

2. Use WEB when the question asks about:
   - current office holders
   - latest launch status or rollout status
   - recent appointments
   - current insurer product wording or current market status
   - any fact that may have changed after the corpus was built
   - WHETHER something has been published, approved, or announced by a regulator
     (e.g. "Has IRDAI published any approved proposals...") — even if a regulation
     name appears in the query, the question is about current publication STATUS

3. Use BOTH only when:
   - the question needs a legal/regulatory baseline from corpus
   - AND also needs current status / latest public update from web
   - e.g. "What are the obligations under X, and are there any recent official
     notifications updating them?" — needs corpus for the base rule, web for updates

Important:
- Do NOT choose BOTH just because you are uncertain.
- BOTH is not a default.
- If unsure between RAG and BOTH, prefer RAG unless the query explicitly needs freshness.
- If unsure between WEB and BOTH, prefer WEB unless the query explicitly asks for regulation/compliance interpretation.

Previous conversation:
{state.get('chat_history', 'None')}

Respond in exactly this format:
ROUTE: <RAG|WEB|BOTH>
CONFIDENCE: <0.00 to 1.00>
REASON: <one short line>
"""
    query = state["query"]
    try:
        response = router_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=query)
        ])
    except Exception as e:
        print(f"⚠️ Router model failed: {e}, falling back to primary")
        response = primary_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=query)
        ])

    raw_output = response.content
    predicted_route, confidence = detect_route_confidence(raw_output)

    fallback_used = False
    fallback_reason = ""

    # ═══════════════════════════════════════════════════════════════
    # Post-router override: catch current-status and update queries
    # that the LLM misclassifies due to regulatory keywords in query.
    # Patterns are semantic signals, NOT copied from test questions.
    # ═══════════════════════════════════════════════════════════════
    query_lower = state["query"].lower()

    # Queries asking WHETHER a regulator has published/approved something
    # are current-status checks → WEB, even if a regulation name appears
    web_status_signals = [
        "has irdai published", "has irdai approved", "has irdai announced",
        "has irdai released", "who is the current", "latest status",
        "has been launched", "rollout status",
    ]
    # Queries needing corpus baseline + recent regulatory updates → BOTH
    both_upgrade_signals = [
        "any recent official", "any updates to", "recent notifications",
        "recent circulars", "recently updated",
    ]

    if predicted_route == "RAG":
        for signal in web_status_signals:
            if signal in query_lower:
                print(f"   ⚠️ Route override: RAG → WEB (signal: '{signal}')")
                predicted_route = "WEB"
                fallback_used = True
                fallback_reason = f"web_status_override:{signal}"
                break

    if predicted_route == "RAG":
        for signal in both_upgrade_signals:
            if signal in query_lower:
                print(f"   ⚠️ Route upgrade: RAG → BOTH (signal: '{signal}')")
                predicted_route = "BOTH"
                fallback_used = True
                fallback_reason = f"both_upgrade_override:{signal}"
                break

    if confidence < 0.6:
        fallback_used = True
        fallback_reason = f"low_router_confidence:{confidence:.2f}"
        predicted_route = "BOTH"

    return {
        "route": predicted_route,
        "router_raw_output": raw_output,
        "router_confidence": confidence,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,
    }


# ═══════════════════════════════════════════════════════════════════
# Dynamic document index — derived from chunk metadata at runtime.
# Maps human-readable doc name variants to their source basename.
# No hardcoded strings: works automatically for any corpus.
# ═══════════════════════════════════════════════════════════════════
def build_doc_index(chunks: list) -> dict:
    index = {}
    seen = set()
    for chunk in chunks:
        src = chunk.metadata.get("source", "")
        basename = os.path.basename(src).replace(".pdf", "")
        if basename in seen:
            continue
        seen.add(basename)
        readable = basename.replace("_", " ").lower()
        index[readable] = basename
        if readable.startswith("the "):
            index[readable[4:]] = basename  # Also match without leading "the"
    return index

DOC_INDEX = build_doc_index(all_chunks)
print(f"✅ Dynamic doc index built: {len(DOC_INDEX)} name variants across {len(set(DOC_INDEX.values()))} documents")


# ═══════════════════════════════════════════════════════════════════
# 🔴 IMPROVEMENT: Query decomposition for multi-part questions
# ═══════════════════════════════════════════════════════════════════
def decompose_query(query: str) -> list:
    """Break multi-part regulatory questions into focused sub-queries."""
    try:
        response = router_llm.invoke([
            SystemMessage(content="""You are a search query optimizer for regulatory document retrieval.
Break the following question into 2-3 focused sub-queries that will help find the relevant document sections.
Each sub-query should target a SPECIFIC fact or clause.
Output one sub-query per line. No numbering, no bullet points, just the queries."""),
            HumanMessage(content=query)
        ])
        sub_queries = [q.strip() for q in response.content.strip().split("\n") if q.strip()]
        return sub_queries[:3]
    except Exception:
        return []


def retrieve_and_rerank(state: AgentState):
    query = state["query"]
    query_lower = query.lower()

    # ── Strict metadata filter: if query names a specific document,
    #    restrict retrieval to that document's chunks only.
    target_doc = None
    for keyword, basename in DOC_INDEX.items():
        if keyword in query_lower:
            target_doc = basename
            break

    # ── Query-aware weights: re-build ensemble per query ────────
    bm25_w, faiss_w = get_retriever_weights(query)
    if bm25_w != 0.35:  # only rebuild if weights differ from default
        _ensemble = EnsembleRetriever(
            retrievers=[bm25_retriever, parent_retriever],
            weights=[bm25_w, faiss_w],
        )
        _comp_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=_ensemble,
        )
    else:
        _comp_retriever = compression_retriever  # reuse default

    # ── Main query: retrieve + rerank against original ──────────
    main_docs = _comp_retriever.invoke(query)
    all_docs = list(main_docs)

    # ── Sub-queries: raw ensemble only (no per-sub reranking) ───
    sub_queries = decompose_query(query)
    print(f"   🔍 Query decomposition: {len(sub_queries)} sub-queries generated")
    for sq in sub_queries:
        try:
            sq_docs = ensemble_retriever.invoke(sq)
            all_docs.extend(sq_docs)
        except Exception:
            pass

    # ── Deduplicate by content fingerprint ──────────────────────
    seen = set()
    unique_docs = []
    for d in all_docs:
        key = d.page_content[:200]
        if key not in seen:
            seen.add(key)
            unique_docs.append(d)

    # ── Strict doc filter (replaces soft boosting) ──────────────
    if target_doc:
        filtered = [d for d in unique_docs
                    if target_doc.lower() in d.metadata.get("source", "").lower()]
        if filtered:
            unique_docs = filtered
            print(f"   📎 Strict doc filter: {len(unique_docs)} chunks from '{target_doc}' only")
        else:
            # Fallback to boost if filter leaves nothing
            non_targeted = [d for d in unique_docs if d not in filtered]
            unique_docs = filtered + non_targeted
            print(f"   📎 Doc filter found 0 results — fallback to full pool")

    # ── Unified final rerank: score all candidates against ORIGINAL query ──
    if len(unique_docs) > 5:
        pairs = [(query, d.page_content) for d in unique_docs]
        scores = _raw_reranker.predict(pairs, show_progress_bar=False)
        ranked = sorted(zip(scores, unique_docs), key=lambda x: x[0], reverse=True)
        unique_docs = [d for _, d in ranked[:5]]
        print(f"   🏆 Unified final rerank: {len(pairs)} candidates → top 5 kept")
    else:
        unique_docs = unique_docs[:5]

    # ── Serialize for state ─────────────────────────────────────
    cleaned_docs = []
    for d in unique_docs:
        cleaned_docs.append({
            "content": d.page_content.strip(),
            "source": d.metadata.get("source", "Unknown"),
            "page": d.metadata.get("page", "Unknown"),
        })

    print(f"   📚 Final context: {len(cleaned_docs)} chunks passed to LLM")
    return {"retrieved_docs": cleaned_docs}

# ==========================================
# grade_and_search (web cache — unchanged)
# ==========================================
import hashlib, json, os

WEB_CACHE_PATH = "/kaggle/working/web_cache.json"

def _load_web_cache() -> dict:
    if os.path.exists(WEB_CACHE_PATH):
        with open(WEB_CACHE_PATH, "r") as f:
            return json.load(f)
    return {}

def _save_web_cache(cache: dict):
    with open(WEB_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)

_web_cache = _load_web_cache()


def grade_and_search(state: AgentState):
    query = state["query"]
    cache_key = hashlib.md5(query.strip().lower().encode()).hexdigest()

    if cache_key in _web_cache:
        cached = _web_cache[cache_key]
        print(f"   📦 Cache hit — using frozen web results ({len(cached['web_results'])} results)")
        return cached

    print(f"\n   [search] Building tiered Tavily search for query...")
    tavily_search = build_tavily_search(query)

    try:
        raw = tavily_search.invoke(query)
        if isinstance(raw, str):
            raw_results = [{"url": "No URL", "content": raw}]
        elif isinstance(raw, list):
            raw_results = []
            for item in raw:
                if isinstance(item, dict):
                    raw_results.append(item)
                elif isinstance(item, str):
                    raw_results.append({"url": "No URL", "content": item})
        else:
            raw_results = []

        filtered_results = []
        for r in raw_results:
            url = r.get("url", "No URL")
            content = r.get("content", "").strip()
            if not content:
                continue
            filtered_results.append({"url": url, "content": content})

        print(f"   ✅ Web search returned {len(filtered_results)} usable results")

        web_context = ""
        for r in filtered_results:
            web_context += f"\n[Source: {r['url']}]\n{r['content']}\n"

        result = {
            "web_results": filtered_results,
            "web_context": web_context.strip(),
        }

    except Exception as e:
        print(f"   ⚠️ Web search failed: {e}")
        result = {"web_results": [], "web_context": ""}

    _web_cache[cache_key] = result
    _save_web_cache(_web_cache)
    print(f"   💾 Web results cached (key: {cache_key[:8]}...)")
    return result


# ═══════════════════════════════════════════════════════════════════
# 🔴 IMPROVEMENT: Simplified, stricter system prompt
# ═══════════════════════════════════════════════════════════════════
def synthesis_and_audit(state: AgentState):
    retrieved_docs = state.get("retrieved_docs", [])
    web_results = state.get("web_results", [])
    query = state["query"]

    # ---- Format RAG context ----
    if isinstance(retrieved_docs, list) and retrieved_docs:
        rag_context_parts = []
        for d in retrieved_docs:
            if isinstance(d, dict):
                content = d.get("content", "").strip()
                source = d.get("source", "Unknown")
                page = d.get("page", "Unknown")
                if content:
                    rag_context_parts.append(f"[Source: {source}, Page: {page}]\n{content}")
            elif isinstance(d, str) and d.strip():
                rag_context_parts.append(d.strip())
        rag_context_text = "\n\n".join(rag_context_parts)
    else:
        rag_context_text = "[NO RAG CONTEXT AVAILABLE]"

    # ---- Format web context ----
    if isinstance(web_results, list) and web_results:
        web_context_parts = []
        for w in web_results:
            if isinstance(w, dict):
                url = w.get("url", "No URL")
                content = w.get("content", "").strip()
                if content:
                    web_context_parts.append(f"[Web: {url}]\n{content}")
        web_context_text = "\n\n".join(web_context_parts)
    else:
        web_context_text = "[NO WEB CONTEXT AVAILABLE]"

    # ── Detect question intent for adaptive output format ───────
    _compliance_signals = [
        "compliant", "non-compliant", "comply", "compliance",
        "violat", "requirement", "obligation", "eligible", "permitted",
        "allowed", "shall", "must", "prohibited", "restrict",
    ]
    _is_compliance_q = any(sig in query.lower() for sig in _compliance_signals)
    _is_web_only = not retrieved_docs or rag_context_text == "[NO RAG CONTEXT AVAILABLE]"

    if _is_compliance_q:
        output_format = """For compliance questions, structure your answer as:
REGULATORY RULE: [Quote or close paraphrase from the source]
CITATION: [Source: document, Page: X]
VERDICT: [COMPLIANT / NON-COMPLIANT / INDETERMINATE]
REASONING: [1–2 sentences connecting rule to the question]"""
    else:
        output_format = """For factual retrieval questions, structure your answer as:
ANSWER: [Direct, precise answer to the question]
CITATION: [Source: document, Page: X — one per factual claim]
NOTES: [Any important caveats or related provisions, if relevant]"""

    web_hardening = ""
    if _is_web_only:
        web_hardening = """
WEB-ONLY RULES (this response uses web context only):
- If a fact is not explicitly confirmed in the retrieved web sources, respond: "No confirmed information found in the retrieved sources for this point."
- Do NOT infer, extrapolate, or fill gaps from general knowledge.
- Limit content from any single web source to 3–4 sentences maximum.
- Prefer official domains (irdai.gov.in, pib.gov.in) over all others.
- If sources conflict, state both positions and cite each.
"""

    system_prompt = f"""You are an IRDAI Regulatory Compliance Analyst. Provide precise, legally-grounded answers using ONLY the evidence provided below.

RESPONSE PROTOCOL:
1. READ all provided context carefully before answering.
2. EXTRACT the specific regulatory rule, clause, or provision that answers the question.
3. CITE every factual claim with [Source: document, Page: X] for RAG or [Web: URL] for web evidence.
4. If the answer is NOT in the provided context, say: "The specific provision is not present in the available documents."

STRICT RULES:
- NEVER state a fact without a citation from the provided context.
- NEVER invent section numbers, clause numbers, committee names, person names, dates, or numerical values.
- NEVER use phrases like "typically", "generally", or "it is common practice" unless directly cited.
- If the query asks for specific numbers (ages, percentages, amounts), either quote the exact number from context or state it is not available.
- Keep the answer focused. Avoid boilerplate sections that don't apply to the question type.
{web_hardening}
OUTPUT FORMAT:
{output_format}

SOURCE PRIORITY:
1. RAG Context (regulatory documents) — primary authority
2. Official IRDAI/government web pages — for current status only
3. Insurer annual reports — for public disclosure questions only
4. Other web sources — only if clearly identified as secondary

RAG Context:
{rag_context_text}

Web Context:
{web_context_text}
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ]

    response = primary_llm.invoke(messages)
    final_text = response.content

    return {"final_answer": final_text}


# ==========================================
# 4. Build Graph (unchanged logic)
# ==========================================
workflow = StateGraph(AgentState)

workflow.add_node("router", intent_router)
workflow.add_node("retrieve", retrieve_and_rerank)
workflow.add_node("search", grade_and_search)
workflow.add_node("synthesize", synthesis_and_audit)

workflow.set_entry_point("router")

def route_workflow(state: AgentState):
    route = state.get("route", "BOTH")
    if route == "RAG":
        return "retrieve"
    elif route == "WEB":
        return "search"
    elif route == "BOTH":
        return "retrieve"
    else:
        return "retrieve"

workflow.add_conditional_edges(
    "router",
    route_workflow,
    {"retrieve": "retrieve", "search": "search"}
)

def after_retrieve_logic(state: AgentState):
    if state.get("route") == "RAG":
        return "synthesize"
    return "search"

workflow.add_conditional_edges(
    "retrieve",
    after_retrieve_logic,
    {"synthesize": "synthesize", "search": "search"}
)

workflow.add_edge("search", "synthesize")
workflow.add_edge("synthesize", END)

app_graph = workflow.compile()
print("✅ Dynamic LangGraph backend compiled and ready.")


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Hybrid retriever ready: BM25(k=22) + ParentDoc-FAISS(k=22) → BGE-reranker-large(top_n=5)
✅ Dynamic doc index built: 21 name variants across 19 documents
✅ Dynamic LangGraph backend compiled and ready.


**Test Execution**

In [ ]:
# Test the pipeline
test_query = """Under the IRDAI Insurance Fraud Monitoring Framework Guidelines 2025, what are the five official categories of fraud, 
                which committee oversees the framework, and which committee operationalises it?"""

inputs = {
    "query": test_query,
    "chat_history": "",
    "route": "",
    "router_raw_output": "",
    "router_confidence": 0.0,
    "fallback_used": False,
    "fallback_reason": "",
    "retrieved_docs": [],
    "web_results": [],
    "web_context": "",
    "final_answer": ""
}

# ── Quick sanity check before running ──────────────────────────────────────
print("🔍 Domain tier preview for this query:")
_ = build_tavily_search(test_query)  # just prints the tier, doesn't search
print()

print("Initiating Audit...\n")
print("-" * 50)
for output in app_graph.stream(inputs):
    for key, value in output.items():
        print(f"🟢 Node executed: {key}")
print("-" * 50)

print("\n📝 FINAL AUDIT REPORT:\n")
print(value["final_answer"])

**Interactive UI for Q&A (add if needed)**

## **DeepEval Setup**

## Configure DeepEval to use Local Qwen2.5-32B-Instruct-AWQ

In [4]:
# ═══════════════════════════════════════════════════════════════════
# Configure DeepEval — Local Qwen2.5-14B-Instruct-AWQ
# ═══════════════════════════════════════════════════════════════════
# Replaces: Qwen2.5-32B-Instruct-AWQ (timed out on T4×2)
# Why 14B?
#   - Fits on a SINGLE T4 GPU (~8GB AWQ) — no cross-GPU PCIe penalty
#   - 2-3× faster inference vs 32B split across 2 GPUs
#   - Still very capable for structured evaluation JSON tasks
#   - Each metric call: ~10-15s vs ~30-60s with 32B
#   - Total eval time: ~30-60 min vs ~2-3 hours
#
# DEPENDENCY CONSTRAINTS (Kaggle T4 environment):
#   - transformers==4.51.3 (native autoawq, no gptqmodel)
#   - gptqmodel uninstalled (missing C++ kernels on Kaggle)
#   - import awq FIRST to force CPU init, bypass meta tensor crash
# ═══════════════════════════════════════════════════════════════════

import torch
import gc
import awq  # ← MUST import before AutoModelForCausalLM to bypass meta tensor bug
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM

# ── Configuration ─────────────────────────────────────────────────
EVAL_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"

# ── Step 1: Load tokenizer ────────────────────────────────────────
print(f"🔄 Loading tokenizer for {EVAL_MODEL_ID}...")
eval_tokenizer = AutoTokenizer.from_pretrained(
    EVAL_MODEL_ID,
    trust_remote_code=True,
)

# ── Step 2: Load model on a SINGLE GPU ────────────────────────────
# 14B-AWQ (~8GB) fits entirely on one T4 (15GB VRAM)
# device_map="auto" will place everything on cuda:0
# No cross-GPU communication = much faster inference!
print(f"🔄 Loading {EVAL_MODEL_ID} (device_map='auto')...")
print(f"   14B-AWQ fits on a single T4 — no cross-GPU penalty!")

eval_model_raw = AutoModelForCausalLM.from_pretrained(
    EVAL_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)

# Show how the model was distributed
if hasattr(eval_model_raw, 'hf_device_map'):
    devices_used = set(str(v) for v in eval_model_raw.hf_device_map.values())
    print(f"✅ Model loaded on device(s): {devices_used}")
else:
    print("✅ Model loaded successfully")

# ── Step 3: Check VRAM usage ──────────────────────────────────────
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    print(f"   GPU {i}: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")

# ── Step 4: Define the DeepEval wrapper ───────────────────────────
class LocalQwenEvalLLM(DeepEvalBaseLLM):
    """
    Local Qwen2.5-14B-Instruct-AWQ for DeepEval evaluation.
    No API calls, no Ollama, no rate limits.
    Fits on a single T4 GPU for fast inference.
    """

    def __init__(self, model, tokenizer):
        self._model = model
        self._tokenizer = tokenizer

    def load_model(self):
        return self._model

    def generate(self, prompt: str) -> str:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a precise evaluation assistant. "
                    "When asked to output JSON, respond with ONLY valid JSON — "
                    "no markdown code fences, no explanatory text, no commentary. "
                    "Just the raw JSON object."
                ),
            },
            {"role": "user", "content": prompt},
        ]

        text = self._tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self._tokenizer(text, return_tensors="pt").to(
            self._model.device if hasattr(self._model, 'device') else 'cuda:0'
        )

        with torch.no_grad():
            output_ids = self._model.generate(
                **inputs,
                max_new_tokens=1024,   # ← Reduced from 2048; DeepEval only needs short JSON
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self._tokenizer.eos_token_id,
            )

        # Decode only the NEW tokens (skip the input prompt)
        generated_ids = output_ids[0][inputs["input_ids"].shape[1] :]
        result = self._tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        # Clean up any markdown code fences that might wrap JSON
        result = result.strip()
        if result.startswith("```json"):
            result = result[7:]
        if result.startswith("```"):
            result = result[3:]
        if result.endswith("```"):
            result = result[:-3]

        return result.strip()

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "Qwen2.5-14B-Instruct-AWQ (local, T4)"


# ── Step 5: Instantiate ──────────────────────────────────────────
eval_llm = LocalQwenEvalLLM(
    model=eval_model_raw,
    tokenizer=eval_tokenizer,
)

print(f"\n✅ DeepEval LLM ready: {eval_llm.get_model_name()}")
print(f"   ℹ️  Running locally — ZERO API rate limits!")
print(f"   ℹ️  Single-GPU inference — much faster than 32B split across 2×T4")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-package

🔄 Loading tokenizer for Qwen/Qwen2.5-14B-Instruct-AWQ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded on device(s): {'cuda:0'}
   GPU 0: 9.3GB allocated, 9.3GB reserved
   GPU 1: 5.4GB allocated, 5.5GB reserved

✅ DeepEval LLM ready: Qwen2.5-14B-Instruct-AWQ (local, T4)
   ℹ️  Running locally — ZERO API rate limits!
   ℹ️  Single-GPU inference — much faster than 32B split across 2×T4


## Load the Q&A Dataset from Excel

In [5]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [6]:
import pandas as pd

EXCEL_PATH = "/kaggle/input/datasets/ashishmm10/qa-dataset/IRDAI_QA_Dataset_v2.xlsx"

qa_df = pd.read_excel(EXCEL_PATH, sheet_name="Dataset")
qa_df.columns = [c.strip().lower() for c in qa_df.columns]

qa_df = qa_df[["q_id", "question", "ground_truth_answer", "expected_route"]].copy()
qa_df.columns = ["q_id", "question", "ground_truth", "route"]

print(f"✅ Loaded {len(qa_df)} Q&A pairs")
print(f"   Route distribution: {qa_df['route'].value_counts().to_dict()}")
display(qa_df[["q_id", "route", "question"]])

✅ Loaded 10 Q&A pairs
   Route distribution: {'RAG': 4, 'WEB': 3, 'BOTH': 3}


,q_id,route,question
0,Q01,RAG,"Under the IRDA Act 1999 Section 5, what is the..."
1,Q02,RAG,Under the IRDAI Insurance Fraud Monitoring Fra...
2,Q03,RAG,Under the Master Circular on Corporate Governa...
3,Q04,RAG,"Under the Master Circular on Rural, Social Sec..."
4,Q05,WEB,"As of official IRDAI sources, what is the late..."
5,Q06,WEB,"Who is the current Chairperson of IRDAI, and w..."
6,Q07,WEB,Has IRDAI published any approved Regulatory Sa...
7,Q08,BOTH,"Under the IRDAI Bima Sugam Regulations 2024, w..."
8,Q09,BOTH,Under the Master Circular on Corporate Governa...
9,Q10,BOTH,"Under the Master Circular on Rural, Social Sec..."


## Run Pipeline & Collect Results

In [ ]:
# # Run this ONCE if the pipeline fails (need to run this before the pipeline!) 
# print("Pre-fetching and caching web results for all questions...")
# for _, row in qa_df.iterrows():
#     state = {"query": row["question"]}
#     grade_and_search(state)   # populates cache
#     time.sleep(1)             # be polite to Tavily rate limits
# print("✅ Web cache populated. Eval runs will be fully reproducible.")

In [7]:
import time

# Updated!
def run_pipeline(question: str) -> dict:
    inputs = {
        "query": question,
        "chat_history": "",
        "route": "",
        "router_raw_output": "",
        "router_confidence": 0.0,
        "fallback_used": False,
        "fallback_reason": "",
        "retrieved_docs": [],
        "web_results": [],
        "web_context": "",
        "final_answer": "",
    }
    final_state = app_graph.invoke(inputs)
    return final_state

# Updated!
def extract_contexts(state: dict) -> list:
    contexts = []

    rag_docs = state.get("retrieved_docs", [])
    if isinstance(rag_docs, list):
        for doc in rag_docs:
            if isinstance(doc, dict):
                content = doc.get("content", "").strip()
                source = doc.get("source", "Unknown")
                page = doc.get("page", "Unknown")
                if content:
                    contexts.append(f"[Source: {source}, Page: {page}]\n{content}")
            elif isinstance(doc, str) and doc.strip():
                contexts.append(doc.strip())

    web_results = state.get("web_results", [])
    if isinstance(web_results, list):
        for item in web_results:
            if isinstance(item, dict):
                url = item.get("url", "No URL")
                content = item.get("content", "").strip()
                if content:
                    contexts.append(f"[Source: {url}]\n{content}")
            elif isinstance(item, str) and item.strip():
                contexts.append(item.strip())

    return contexts if contexts else ["No context retrieved."]


# ── Run pipeline on all 10 questions ──────────────────────────────────────
# Updated!
results = []

for _, row in qa_df.iterrows():
    q_id = row["q_id"]
    question = row["question"]
    expected_route = row["route"]

    print(f"🔄 Running {q_id} [expected={expected_route}]: {question[:70]}...")
    start = time.time()

    try:
        state = run_pipeline(question)
        answer = state.get("final_answer", "").strip()
        contexts = extract_contexts(state)
        elapsed = round(time.time() - start, 1)

        actual_route = state.get("route", "?")
        fallback_used = state.get("fallback_used", False)
        router_confidence = state.get("router_confidence", 0.0)

        print(
            f"   ✅ {elapsed}s | expected={expected_route} | actual={actual_route} "
            f"| fallback={fallback_used} | conf={router_confidence:.2f} "
            f"| ctx_chunks={len(contexts)} | ans_len={len(answer)} chars"
        )

    except Exception as e:
        print(f"   ❌ Pipeline error: {e}")
        answer = ""
        contexts = ["Pipeline error — no context retrieved."]
        actual_route = "ERROR"
        fallback_used = False
        router_confidence = 0.0
        state = {}

    results.append({
        "q_id": q_id,
        "question": question,
        "ground_truth": row["ground_truth"],
        "answer": answer,
        "contexts": contexts,
        "expected_route": expected_route,
        "actual_route": actual_route,
        "fallback_used": fallback_used,
        "fallback_reason": state.get("fallback_reason", ""),
        "router_confidence": router_confidence,
        "router_raw_output": state.get("router_raw_output", ""),
    })

    time.sleep(15) # ✅ Gives Groq's TPM limit time to cool down

print(f"\n✅ Pipeline completed for all {len(results)} questions.")

🔄 Running Q01 [expected=RAG]: Under the IRDA Act 1999 Section 5, what is the term of office for the ...
   ⚖️  Clause-specific query → BM25 0.6 / FAISS 0.4
   🔍 Query decomposition: 3 sub-queries generated
   📎 Strict doc filter: 14 chunks from 'The_IRDA_Act_1999' only
   🏆 Unified final rerank: 14 candidates → top 5 kept
   📚 Final context: 5 chunks passed to LLM
   ✅ 5.8s | expected=RAG | actual=RAG | fallback=False | conf=1.00 | ctx_chunks=5 | ans_len=484 chars
🔄 Running Q02 [expected=RAG]: Under the IRDAI Insurance Fraud Monitoring Framework Guidelines 2025, ...
   ⚖️  Semantic query → BM25 0.35 / FAISS 0.65
   🔍 Query decomposition: 3 sub-queries generated
   🏆 Unified final rerank: 35 candidates → top 5 kept
   📚 Final context: 5 chunks passed to LLM
   ✅ 7.8s | expected=RAG | actual=RAG | fallback=False | conf=0.95 | ctx_chunks=5 | ans_len=1482 chars
🔄 Running Q03 [expected=RAG]: Under the Master Circular on Corporate Governance for Insurers 2024, w...
   ⚖️  Semantic query → BM

## Build DeepEval test cases from the results list:

In [8]:
# ═══════════════════════════════════════════════════════════════════
# 🔴 IMPROVEMENT: Route-aware test case builder
#    Tags each test case with its expected route so we can apply
#    the correct metric set during evaluation.
# ═══════════════════════════════════════════════════════════════════
from deepeval.test_case import LLMTestCase

test_cases = []
test_routes = []  # Track expected route for each test case

for r in results:
    tc = LLMTestCase(
        input=r["question"],
        actual_output=r["answer"],
        retrieval_context=r["contexts"],
        expected_output=r["ground_truth"],
    )
    test_cases.append(tc)
    test_routes.append(r["expected_route"])  # "RAG", "WEB", or "BOTH"

print(f"✅ Built {len(test_cases)} DeepEval test cases")
print(f"   Route breakdown: RAG={test_routes.count('RAG')}, WEB={test_routes.count('WEB')}, BOTH={test_routes.count('BOTH')}")


✅ Built 10 DeepEval test cases
   Route breakdown: RAG=4, WEB=3, BOTH=3


## Define metrics:

In [9]:
# ═══════════════════════════════════════════════════════════════════
# 🔴 IMPROVEMENT: Route-aware metrics
#    - RAG / BOTH questions: all 4 metrics
#    - WEB-only questions: only Faithfulness + Answer Relevancy
#      (Contextual Precision/Recall are meaningless when there's no
#       corpus retrieval — they'll always fail for web-only queries)
# ═══════════════════════════════════════════════════════════════════

from deepeval.metrics import (
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    AnswerRelevancyMetric,
)

# ── Full Metrics (for RAG and BOTH routes) ─────────────────────────
faithfulness_metric = FaithfulnessMetric(
    threshold=0.5,
    model=eval_llm,
    include_reason=True,
    verbose_mode=False,
)

context_precision_metric = ContextualPrecisionMetric(
    threshold=0.5,
    model=eval_llm,
    include_reason=True,
    verbose_mode=False,
)

context_recall_metric = ContextualRecallMetric(
    threshold=0.5,
    model=eval_llm,
    include_reason=True,
    verbose_mode=False,
)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.5,
    model=eval_llm,
    include_reason=True,
    verbose_mode=False,
)

# ── Two metric sets ────────────────────────────────────────────────
RAG_METRICS = [
    faithfulness_metric,
    context_precision_metric,
    context_recall_metric,
    answer_relevancy_metric,
]

WEB_METRICS = [
    faithfulness_metric,
    answer_relevancy_metric,
]

print(f"✅ RAG/BOTH metrics: {[type(m).__name__ for m in RAG_METRICS]}")
print(f"✅ WEB-only metrics: {[type(m).__name__ for m in WEB_METRICS]}")


✅ RAG/BOTH metrics: ['FaithfulnessMetric', 'ContextualPrecisionMetric', 'ContextualRecallMetric', 'AnswerRelevancyMetric']
✅ WEB-only metrics: ['FaithfulnessMetric', 'AnswerRelevancyMetric']


## Run evaluation in batches with cooldown:

In [10]:
# ══════════════════════════
# Run evaluation in batches 
# ══════════════════════════
import time
import pandas as pd
from deepeval import evaluate as deepeval_evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

# ── CRITICAL: Set timeouts BEFORE calling evaluate() ─────────────
import os
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
os.environ["ERROR_REPORTING"] = "NO"

# DeepEval timeout overrides (default is 180s — too short for local models)
# 14B is ~2-3x faster than 32B, but still slower than API calls
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "1200"    # 20 min per test case (all metrics combined)
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "600"  # 10 min per individual LLM call
os.environ["DEEPEVAL_LOG_STACK_TRACES"] = "1"                        # Full tracebacks if something fails

BATCH_COOLDOWN = 10    # Pause between batches (GPU memory cleanup)
RETRY_COOLDOWN = 30    # Wait before retrying a failed batch
MAX_RETRIES = 2        # Retry failed batches up to 2 times

print("🚀 Starting DeepEval evaluation (Qwen2.5-14B-Instruct-AWQ local, route-aware)...")
print(f"   ⏱️  Per-task timeout: {os.environ['DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE']}s")
print(f"   ⏱️  Per-attempt timeout: {os.environ['DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE']}s")

batch_size = 1
all_results_eval = []

for i in range(0, len(test_cases), batch_size):
    batch = test_cases[i : i + batch_size]
    batch_route = test_routes[i]
    batch_num = i // batch_size + 1

    # Select metrics based on route
    if batch_route == "WEB":
        metrics = WEB_METRICS
        metrics_label = "WEB (Faith+Relevancy only)"
    else:
        metrics = RAG_METRICS
        metrics_label = "RAG/BOTH (all 4 metrics)"

    print(f"\n📦 Evaluating batch {batch_num} (Q{i+1}) [{metrics_label}]...")

    success = False
    for attempt in range(1, MAX_RETRIES + 2):  # +2 because range is exclusive
        try:
            eval_results = deepeval_evaluate(
                test_cases=batch,
                metrics=metrics,
                async_config=AsyncConfig(
                    run_async=False,
                    max_concurrent=1,
                ),
                display_config=DisplayConfig(
                    print_results=True,
                    show_indicator=True,
                ),
                error_config=ErrorConfig(
                    ignore_errors=True,  # Don't crash on JSON parse errors
                ),
            )
            all_results_eval.extend(eval_results.test_results)
            print(f"   ✅ Batch {batch_num} done (attempt {attempt}).")
            success = True
            break

        except Exception as e:
            print(f"   ⚠️  Batch {batch_num} error (attempt {attempt}): {e}")
            if attempt <= MAX_RETRIES:
                print(f"   🔄 Retrying in {RETRY_COOLDOWN}s...")
                time.sleep(RETRY_COOLDOWN)
            else:
                print(f"   ❌ Batch {batch_num} failed after {MAX_RETRIES + 1} attempts. Skipping.")

    # GPU memory cleanup between batches
    if i + batch_size < len(test_cases):
        import torch, gc
        torch.cuda.empty_cache()
        gc.collect()
        print(f"   ⏳ Cooling down {BATCH_COOLDOWN}s (GPU cache cleared)...")
        time.sleep(BATCH_COOLDOWN)

print(f"\n✅ Full DeepEval evaluation complete!")
print(f"   📊 Collected results for {len(all_results_eval)} out of {len(test_cases)} test cases.")


🚀 Starting DeepEval evaluation (Qwen2.5-14B-Instruct-AWQ local, route-aware)...
   ⏱️  Per-task timeout: 1200s
   ⏱️  Per-attempt timeout: 600s

📦 Evaluating batch 1 (Q1) [RAG/BOTH (all 4 metrics)]...


✨ You're running DeepEval's latest Faithfulness Metric! (using Qwen2.5-14B-Instruct-AWQ (local, T4), strict=False,
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using Qwen2.5-14B-Instruct-AWQ (local, T4), 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using Qwen2.5-14B-Instruct-AWQ (local, T4), 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Qwen2.5-14B-Instruct-AWQ (local, T4), 
strict=False, async_mode=False)...

...

Output()



Metrics Summary

  - ✅ Faithfulness (score: 0.875, threshold: 0.5, strict: False, evaluation model: Qwen2.5-14B-Instruct-AWQ (local, T4), reason: The score is 0.88 because the actual output incorrectly presents the minimum percentage increase as 12.50% instead of 12.5% for the specified market-share band., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: Qwen2.5-14B-Instruct-AWQ (local, T4), reason: The score is 1.00 because all the nodes in the retrieval contexts are relevant to the input, ensuring high precision without any irrelevant nodes., error: None)
  - ✅ Contextual Recall (score: 0.75, threshold: 0.5, strict: False, evaluation model: Qwen2.5-14B-Instruct-AWQ (local, T4), reason: The score is 0.75 because the expected output accurately reflects the information provided in the 1st node of the retrieval context, including the minimum percentage increases for different market share bands and their applicability to general insu

⚠ WARNING: No hyperparameters logged.
» ]8;id=332392;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1057.98s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

   ✅ Batch 10 done (attempt 1).

✅ Full DeepEval evaluation complete!
   📊 Collected results for 10 out of 10 test cases.


## Parse results into a DataFrame:

In [11]:
# ═══════════════════════════════════════════════════════════════════
# Results export — updated for route-aware evaluation
# ═══════════════════════════════════════════════════════════════════
rows = []

for idx, (r, tc_result) in enumerate(zip(results, all_results_eval)):
    row = {
        "q_id": r["q_id"],
        "expected_route": r["expected_route"],
        "actual_route": r["actual_route"],
        "fallback_used": r["fallback_used"],
        "fallback_reason": r["fallback_reason"],
        "router_confidence": r["router_confidence"],
        "question": r["question"],
    }

    for metric_result in tc_result.metrics_data:
        col = metric_result.name.lower().replace(" ", "_")
        row[col] = metric_result.score if metric_result.success is not None else float("nan")
        row[f"{col}_reason"] = metric_result.reason

    rows.append(row)

scores_df = pd.DataFrame(rows)

# These columns may not exist for WEB-only questions — handle gracefully
metric_cols = ["faithfulness", "contextual_precision", "contextual_recall", "answer_relevancy"]
available_cols = [c for c in metric_cols if c in scores_df.columns]

print("\n📊 Per-Question Scores:")
display(scores_df[["q_id", "expected_route", "actual_route", "fallback_used"] + available_cols])

print("\n🏆 Overall Mean Scores (all questions):")
print(scores_df[available_cols].mean(skipna=True).round(3))

# Also show means split by route type
print("\n📈 Scores by Route Type:")
for route in ["RAG", "WEB", "BOTH"]:
    subset = scores_df[scores_df["expected_route"] == route]
    if len(subset) > 0:
        print(f"\n  {route} ({len(subset)} questions):")
        print(f"  {subset[available_cols].mean(skipna=True).round(3).to_dict()}")

scores_df.to_excel('/kaggle/working/deepeval_scores_v3.xlsx', index=False)
print("\n✅ File saved to /kaggle/working/deepeval_scores_v3.xlsx")



📊 Per-Question Scores:


,q_id,expected_route,actual_route,fallback_used,faithfulness,contextual_precision,contextual_recall,answer_relevancy
0,Q01,RAG,RAG,False,1.000000,1.00,1.000000,1.000000
1,Q02,RAG,RAG,False,0.714286,1.00,1.000000,1.000000
2,Q03,RAG,RAG,False,1.000000,1.00,1.000000,1.000000
3,Q04,RAG,RAG,False,1.000000,0.75,1.000000,1.000000
4,Q05,WEB,WEB,False,0.800000,NaN,NaN,1.000000
5,Q06,WEB,WEB,False,1.000000,NaN,NaN,1.000000
6,Q07,WEB,WEB,False,1.000000,NaN,NaN,0.333333
7,Q08,BOTH,BOTH,False,0.600000,1.00,0.666667,1.000000
8,Q09,BOTH,BOTH,False,0.750000,1.00,0.500000,0.500000
9,Q10,BOTH,BOTH,False,0.875000,1.00,0.750000,0.875000



🏆 Overall Mean Scores (all questions):
faithfulness            0.874
contextual_precision    0.964
contextual_recall       0.845
answer_relevancy        0.871
dtype: float64

📈 Scores by Route Type:

  RAG (4 questions):
  {'faithfulness': 0.929, 'contextual_precision': 0.938, 'contextual_recall': 1.0, 'answer_relevancy': 1.0}

  WEB (3 questions):
  {'faithfulness': 0.933, 'contextual_precision': nan, 'contextual_recall': nan, 'answer_relevancy': 0.778}

  BOTH (3 questions):
  {'faithfulness': 0.742, 'contextual_precision': 1.0, 'contextual_recall': 0.639, 'answer_relevancy': 0.792}

✅ File saved to /kaggle/working/deepeval_scores_v3.xlsx
